# 55. Matched BatchNorm statistics on every stream

Everything the study reports about effective rank, srank and dormant units is
read through one fixed set of BatchNorm statistics: a mean of zero and a
variance of one at every layer, taken from a freshly built ResNet50 and never
updated. Notebook 52 showed that on ordering A this choice decides the
direction of travel, not only the size of the numbers. It was run on five of
the fifteen streams.

This notebook runs the corrected reading on all fifteen, and it does a second
job in the same pass.

**Part 1, the matched reading.** Each stream is read at three points: boundary
0, which is the untrained generator, boundary 9, the end of the nine-group
stream, and boundary 10, after one learned group has been unlearned. Each point
is read twice, through the fixed statistics and through statistics measured
from the model's own weights. Three of the paper's correlation tables are
computed from exactly these points, so this is what decides whether sections
3.3 to 3.5 survive the correction.

**Part 2, the rescaling control under matched statistics.** Notebook 46 found
that most of the rank change at the unlearn is explained by the weights simply
getting smaller, read through the fixed statistics. Matched statistics are
measured from the weights themselves, so in principle they should divide the
scale back out and the whole effect should vanish. If it does, the rescaling
confound was another face of the protocol. If it does not, it is a separate
problem. Nobody has looked.

**Why one notebook.** Both parts read the same thirty checkpoint files, 16.2 GB
in all. Run as two notebooks that is 32 GB of downloading. Here each stream's
pair comes down once, both parts read it, and the files are deleted before the
next stream starts, so about 1.1 GB sits on disk at a time.

**No training happens here.** Every number is a forward pass over saved
weights.

## Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules

# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/sumitasthana/PATH.git", str(REPO)],
                       check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package],
                       check=True)
sys.path.insert(0, str(REPO))

import torch
from torch import nn
from torch.func import functional_call
from tqdm.auto import tqdm
from uncle import checkpoint
from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import HyperNetwork, build_target
from uncle.plasticity_indicators import (
    SRANK_DELTA, THRESHOLDS, activation_sites, dormant_fraction,
    fixed_image_sets, flatten, reference_code)
from uncle.probe import _features_and_logits
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.trainer import UnCLe

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                               text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Running at:", HEAD)
print("Device:", GPU_NAME)

## Settings

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# All fifteen streams. Ordering A comes first on purpose: it is the one with
# published readings, so the sanity gate below can check this machine against
# them before the other ten streams are started.
READ_ORDERS = ["order_A", "order_B", "order_C"]
READ_SEEDS = [0, 1, 2, 3, 4]

# Boundary 0 is rebuilt from the seed and needs no file. Boundary 9 is the end
# of the nine-group stream and boundary 10 is after the unlearn; each is one
# 539 MB file per stream.
READ_BOUNDARIES = [0, 9, 10]

# The task each ordering unlearns at the end. Part of what names a run, so the
# rebuilt configuration has to carry it.
FORGET = {"order_A": "0", "order_B": "4", "order_C": "8"}

# The pictures the indicators are read on. The same 256 every published reading
# in this study used, drawn from the held-out split.
IMAGES_PER_SET = 256

# The pictures the statistics are measured on. These come from the training
# split of the nine learned groups, so they do not overlap the 256 above. 2048
# is 32 batches of 64, enough for a stable mean and variance at every layer.
ADAPT_IMAGES = 2048
BATCH_SIZE = 64

# Only the fixed reference code is read. Every published indicator number uses
# it, and the correlation tables this run feeds are all computed from it.
# Notebook 52 swept every task code as well; that was its own question, it was
# answered on ordering A, and repeating it here would multiply the work by ten
# for nothing this run needs.
#
# The pre-ReLU reading is left out for the same reason. It belongs to appendix
# B's question about where the collapse happens, which is deliberately on
# ordering A alone.

PART1_KEY = f"uncle/indicator_protocol/{STUDY_ID}/matched_statistics_v1.jsonl"
PART2_KEY = f"uncle/indicator_protocol/{STUDY_ID}/rescaling_matched_v1.jsonl"

# Ordering A's published readings, from notebook 52's file. The gate below
# checks this machine against them. These are the mean over the five ordering A
# seeds of the effective rank of the 2048 inputs to the final classifier, read
# on the in-stream pictures with the reference code.
GATE = {(0, "start"): 5.5631, (0, "own"): 106.2719,
        (9, "start"): 2.7886, (9, "own"): 143.0007}

# How far a reading may sit from the published one before the run stops. One
# percent, not the 1e-3 an earlier draft of this plan asked for. Two runs of
# the control network under identical code were compared on 20 November and
# differed by a median of 4 to 11 percent at every trained boundary, from
# nothing but GPU nondeterminism. These readings are forward passes over saved
# weights rather than fresh training, so they should land far closer than that,
# but 1e-3 on a mean of 5.5631 is 0.018 percent, which a different GPU can miss
# without anything being wrong. One percent is loose enough to survive a change
# of machine and far tighter than any difference that would change a
# conclusion.
GATE_TOLERANCE = 0.01

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
CACHE = Path("/content/matched") if IN_COLAB else REPO / "outputs/matched-statistics"
CACHE.mkdir(parents=True, exist_ok=True)

READ_RUNS = [(o, s) for o in READ_ORDERS for s in READ_SEEDS]
downloads = len(READ_RUNS) * len([p for p in READ_BOUNDARIES if p > 0])
print(f"{len(READ_RUNS)} streams x {len(READ_BOUNDARIES)} boundaries")
print(f"{downloads} model files to download, about {downloads * 0.539:.1f} GB in all,")
print("about 1.1 GB on disk at a time. No training. Forward passes only.")

## Connect, and check the models are there

In [ ]:
def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip(chr(34)).strip(chr(39))
    return value


store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))

# The settings the streams were trained with, read back rather than retyped, so
# the model this notebook rebuilds has the shape its checkpoint expects.
plan = json.loads(store.download(
    f"uncle/indicators/{STUDY_ID}/job2_summary.json", CACHE).read_text(encoding="utf-8"))
CLASSES = plan["classes_per_task"]
print("Streams used learning rate", plan["learning_rate"], "| epochs", plan["epochs"],
      "| batch", plan["batch_size"], "| classes per task", CLASSES)

# Fail now rather than ten minutes in, if a model this session needs is missing.
missing = []
for order, seed in READ_RUNS:
    for point in READ_BOUNDARIES:
        if point == 0:
            continue
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        if store.info(key) is None:
            missing.append(key)
if missing:
    raise RuntimeError("Missing models:\n" + "\n".join(sorted(set(missing))))
print("Every model this session needs is in R2.")

## The pictures

In [ ]:
from torch.utils.data import ConcatDataset

STREAM_TASKS = [str(i) for i in range(9)]
needed = tuple(STREAM_TASKS + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

# The reading sets. in_stream is pictures from groups the models learned, and
# probe is the held-out group 9 that no model ever trained on.
sources = {
    "in_stream": ConcatDataset([tasks[name]["test"] for name in STREAM_TASKS]),
    "probe": tasks[PROBE_TASK]["test"],
}
image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)

# The adaptation set, from the training split of the same nine groups. The
# drawing is seeded inside fixed_image_sets, so every seed and every boundary
# measures its statistics on exactly the same pictures and only the model
# differs.
adapt_source = ConcatDataset([tasks[name]["train"] for name in STREAM_TASKS])
adapt_images = fixed_image_sets({"adapt": adapt_source},
                                count=ADAPT_IMAGES)["adapt"]

for name, images in image_sets.items():
    print(f"read   {name:<10} {tuple(images.shape)}")
print(f"adapt  {'train':<10} {tuple(adapt_images.shape)}")

## The readers

These are notebook 52's, unchanged, so the numbers this run produces go through
the same arithmetic as the published ones. `build` takes an already downloaded
file instead of fetching its own, because this notebook reads each file several
times before deleting it.

In [ ]:
import random
import numpy as np


def build(order, seed, point, path=None):
    """The model this stream had at `point`. Point 0 needs no file.

    Point 0 is the untrained generator. It is rebuilt from the seed rather than
    loaded, which is why no boundary-0 file exists anywhere in this study.
    """
    settings = study_defaults(order)
    settings["requests"] = settings["requests"] + (("forget", FORGET[order]),)
    config = Config(**settings, seed=seed,
                    learning_rate=plan["learning_rate"], epochs=plan["epochs"],
                    batch_size=plan["batch_size"],
                    initialization=plan["initialization"], device=DEVICE)
    # The generator's starting weights come from the global random number
    # generator, not from config.seed by itself: HyperNetwork.__init__ calls
    # kaiming_normal_, normal_ and randn, and all three read the global state.
    # uncle/paired_study.py seeds these before it builds, so a rebuilt model
    # only matches the study's unless this does the same. At a loaded boundary
    # the checkpoint overwrites all of it, so this matters only for point 0.
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(config.seed)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if point > 0:
        if path is None:
            key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
            path = store.download(key, CACHE)
        payload = torch.load(path, map_location=DEVICE, weights_only=False)
        checkpoint.restore(payload, hypernet=hypernet, uncle=trainer)
        del payload
    return target, hypernet, trainer, path

In [ ]:
def rank_pair(matrix, centre):
    """Effective rank and srank of one feature matrix.

    `matrix` has one row per picture and 2048 columns. Setting `centre` to True
    subtracts the average row first.

    Effective rank is the exponential of the entropy of the normalised singular
    values, from Roy and Vetterli 2007. srank counts how many singular values
    are needed to reach 99 percent of their total, from Kumar et al. 2021.
    Both are copied from the library rather than imported so that both buffer
    routes go through exactly the same arithmetic.
    """
    work = matrix.double()
    if centre:
        work = work - work.mean(dim=0, keepdim=True)
    values = torch.linalg.svdvals(work)
    total = values.sum()
    if total <= 0:
        return float("nan"), 0
    weights = values / total
    weights = weights[weights > 0]
    effective = float(torch.exp(-(weights * weights.log()).sum()).item())
    shares = torch.cumsum(values, 0) / total
    return effective, int((shares < 1 - SRANK_DELTA).sum().item()) + 1


def mean_share(matrix):
    """How much of the matrix is the one row every picture shares.

    Take the average row, measure its length, and compare that to the length of
    the whole matrix. The square root of the number of rows is there because
    the whole matrix stacks that many rows. The result runs from 0 to 1. Near 1
    means every picture gives almost the same features, which by itself drives
    the uncentred effective rank towards 1 whatever else the model does.
    """
    work = matrix.double()
    norm = float(work.norm())
    if norm <= 0:
        return float("nan")
    return float(work.mean(dim=0).norm() * (work.shape[0] ** 0.5) / norm)


@torch.no_grad()
def measured_buffers(target, trainer, weights):
    """BatchNorm statistics that match these generated weights.

    The study reads every model through `trainer.buffer_template`, which holds
    what a freshly built ResNet50 has: a running mean of 0 and a running
    variance of 1 at every BatchNorm layer, values that never change however
    far the weights travel. This function makes the other reading. It runs the
    generated weights over the adaptation pictures with the BatchNorm layers in
    training mode, so each layer measures the mean and variance its own weights
    actually produce on data from the learned groups.

    Setting `momentum` to None makes each layer keep a plain running average
    over every batch it sees, instead of an exponentially decaying one. The
    result then does not depend on the order the batches arrive in, which an
    exponential average would. The momentum values are put back afterwards.
    """
    buffers = {name: value.detach().clone()
               for name, value in trainer.buffer_template.items()}
    # Start every counter at zero so the running average covers these batches
    # and nothing else.
    for name, value in buffers.items():
        if name.endswith("num_batches_tracked"):
            value.zero_()

    saved = {}
    for name, module in target.named_modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            saved[name] = module.momentum
            module.momentum = None

    was_training = target.training
    target.train()
    try:
        for start in range(0, len(adapt_images), BATCH_SIZE):
            batch = adapt_images[start:start + BATCH_SIZE].to(trainer.device)
            # strict=True means every parameter and buffer must be supplied, so
            # a name that stops matching fails here rather than silently using
            # the module's own value.
            functional_call(target, (weights, buffers), (batch,), strict=True)
    finally:
        target.train(was_training)
        for name, module in target.named_modules():
            if name in saved:
                module.momentum = saved[name]

    # Every BatchNorm layer should have seen every batch. If this fails, the
    # statistics are not an average over the data they claim to describe.
    counted = {int(value.item()) for name, value in buffers.items()
               if name.endswith("num_batches_tracked")}
    expected = (len(adapt_images) + BATCH_SIZE - 1) // BATCH_SIZE
    assert counted == {expected}, (counted, expected)
    return buffers


@torch.no_grad()
def read_dormant(hypernet, target, trainer, images, weights, buffers):
    """The share of units that have gone quiet, for one set of weights and buffers.

    A unit is called dormant when its mean absolute activation is small beside
    the mean over all units at the same place in the network. Sokar et al. 2023
    define it that way and report three cut-offs, which is what THRESHOLDS
    holds. This is the third indicator in the paper's own opening table, and
    the only one that had never been read through both buffer routes.

    The counting itself is the library's, so these numbers go through exactly
    the same arithmetic as the published ones. The one difference is that the
    buffers are supplied by the caller instead of always being the fixed
    template, which is the whole point of this notebook.

    Unlike effective rank, this reads every ReLU in the network rather than
    only the 2048 numbers at the classifier. If the fixed template is what
    makes units look dormant, it should show here most of all: a unit counts as
    dormant when its activation is small, and reading through statistics that
    do not match the weights is exactly what shrinks activations.
    """
    modes = hypernet.training, target.training
    hypernet.eval()
    target.eval()
    try:
        def run(reset):
            for start in range(0, len(images), BATCH_SIZE):
                # The sums are keyed by the position of each ReLU call, so the
                # counter has to go back to zero at the start of every batch.
                reset()
                _features_and_logits(hypernet, target, trainer, None,
                                     images[start:start + BATCH_SIZE].to(trainer.device),
                                     weights=weights, buffers=buffers)
        return dormant_fraction(activation_sites(target, run))
    finally:
        hypernet.train(modes[0])
        target.train(modes[1])


@torch.no_grad()
def read_features(hypernet, target, trainer, images, weights, buffers):
    """The classifier's 2048 inputs, for one set of weights and one set of buffers.

    Evaluation mode is forced. A BatchNorm layer left in training mode rescales
    using whatever batch is in front of it, which revives a nearly dead signal
    and hides what is being measured. The previous modes are put back after.
    """
    modes = hypernet.training, target.training
    hypernet.eval()
    target.eval()
    collected = []
    try:
        for start in range(0, len(images), BATCH_SIZE):
            batch = images[start:start + BATCH_SIZE].to(trainer.device)
            # task is unused once weights and buffers are both supplied.
            captured, _ = _features_and_logits(hypernet, target, trainer, None,
                                               batch, weights=weights,
                                               buffers=buffers)
            collected.append(captured.float().cpu())
    finally:
        hypernet.train(modes[0])
        target.train(modes[1])
    return torch.cat(collected)

## Part 1: the matched reading

One model, one code, two routes, two picture sets. The fixed route is the
protocol the study uses everywhere. The matched route measures the statistics
from the same generated weights that are about to be read, which is the only
way the two readings differ.

In [ ]:
def read_indicators(order, seed, point, path):
    """The indicator readings for one saved model, through both routes."""
    target, hypernet, trainer, _ = build(order, seed, point, path)
    try:
        code_vec = reference_code(hypernet)
        weights = hypernet.weights_from_code(code_vec)
        # The size of the generated weights. Part 2 needs this, and recording
        # it here means part 2 does not have to compute it a second time.
        norm = float(flatten(hypernet, code_vec)["target"].norm())
        routes = {"start": trainer.buffer_template,
                  "own": measured_buffers(target, trainer, weights)}
        rows = []
        for mode, buffers in routes.items():
            for set_name, images in image_sets.items():
                matrix = read_features(hypernet, target, trainer, images,
                                       weights, buffers)
                assert matrix.shape == (IMAGES_PER_SET, 2048), matrix.shape
                raw_e, raw_s = rank_pair(matrix, centre=False)
                mid_e, mid_s = rank_pair(matrix, centre=True)
                quiet = read_dormant(hypernet, target, trainer, images,
                                     weights, buffers)
                rows.append(dict(
                    study=STUDY_ID, order=order, seed=seed, point=point,
                    code="reference", buffers=mode, images=set_name,
                    effective_rank=raw_e, srank=raw_s,
                    effective_rank_centred=mid_e, srank_centred=mid_s,
                    mean_share=mean_share(matrix), **quiet,
                    weight_norm=norm,
                    n_images=int(matrix.shape[0]),
                    n_adapt_images=int(len(adapt_images)),
                    code_revision=HEAD, device=GPU_NAME))
        return rows, norm
    finally:
        del trainer, hypernet, target
        import gc
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

## Part 2: the rescaling control, through both routes

The unlearn makes the generated weights smaller. Notebook 46 asked how much of
the rank change at the unlearn is explained by that shrinkage alone, and
answered it with two extra arms: the post-unlearn weights grown back to their
former size, and the intact weights shrunk to the size the unlearn left. Both
were read through the fixed statistics.

Here each arm is read through both routes. The matched statistics are measured
from the scaled weights, not from the unscaled ones. That is the honest
comparison and it is also the thing being tested: BatchNorm divides by a
measured spread, so statistics taken from scaled weights should divide the
scale straight back out. If the matched arms land on top of each other while
the fixed arms separate, the rescaling result was the protocol again.

In [ ]:
def read_rescaling(order, seed, paths, norms):
    """The four arms of the rescaling control, each through both routes.

    `up` grows the post-unlearn weights back to the size they had before the
    unlearn. `down` shrinks the intact weights to the size the unlearn left.
    The two unscaled arms are the models as they are.
    """
    up = norms[9] / norms[10]
    down = norms[10] / norms[9]
    plan_arms = {9: [("b9", 1.0), ("b9_at_b10_norm", down)],
                 10: [("b10", 1.0), ("b10_at_b9_norm", up)]}

    rows = []
    for point, arms in plan_arms.items():
        target, hypernet, trainer, _ = build(order, seed, point, paths[point])
        try:
            code_vec = reference_code(hypernet)
            base = hypernet.weights_from_code(code_vec)
            for label, scale in arms:
                weights = ({name: value * scale for name, value in base.items()}
                           if scale != 1.0 else base)
                routes = {"start": trainer.buffer_template,
                          "own": measured_buffers(target, trainer, weights)}
                for mode, buffers in routes.items():
                    for set_name, images in image_sets.items():
                        matrix = read_features(hypernet, target, trainer,
                                               images, weights, buffers)
                        e, s = rank_pair(matrix, centre=False)
                        rows.append(dict(
                            study=STUDY_ID, order=order, seed=seed,
                            arm=label, source=point, scale=scale,
                            buffers=mode, images=set_name,
                            effective_rank=e, srank=s,
                            mean_share=mean_share(matrix),
                            norm_b9=norms[9], norm_b10=norms[10],
                            norm_ratio=norms[10] / norms[9],
                            n_images=int(matrix.shape[0]),
                            code_revision=HEAD, device=GPU_NAME))
        finally:
            del trainer, hypernet, target
            import gc
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    return rows

## The sanity gate

Ordering A is read first, and when its five streams are done the four published
means are checked before the other ten streams start. A failure here means this
machine is not reading what the study read, and nothing after it would be
comparable, so the run stops.

In [ ]:
def check_gate(rows):
    """Compare ordering A against its published means. Raises if it is off.

    What is compared: the effective rank of the 2048 numbers entering the final
    classifier, on the in-stream pictures, with the reference code, averaged
    over the five ordering A seeds. Four readings, two boundaries by two
    routes. The individual seeds are not checked. They spread very widely at
    boundary 0, from 3.13 to 7.14 on the published run, so only the mean is
    stable enough to check anything against.
    """
    print("SANITY GATE, ordering A, effective rank at the classifier,")
    print("mean over five seeds, in-stream pictures, reference code")
    worst = 0.0
    for (point, mode), published in sorted(GATE.items()):
        here = [r["effective_rank"] for r in rows
                if r["order"] == "order_A" and r["point"] == point
                and r["buffers"] == mode and r["images"] == "in_stream"]
        assert len(here) == len(READ_SEEDS), (point, mode, len(here))
        got = sum(here) / len(here)
        off = abs(got / published - 1.0)
        worst = max(worst, off)
        route = "fixed" if mode == "start" else "matched"
        print(f"  boundary {point}, {route:>7} statistics: "
              f"published {published:10.4f}   here {got:10.4f}   "
              f"off by {100 * off:.3f}%")
    if worst > GATE_TOLERANCE:
        raise RuntimeError(
            f"ordering A is off by {100 * worst:.3f}%, past the "
            f"{100 * GATE_TOLERANCE:.0f}% this gate allows. This machine is not "
            f"reading what the study read, so the other ten streams would not "
            f"be comparable. Stop and find out why before going on.")
    print(f"  passed, worst gap {100 * worst:.3f}% of "
          f"{100 * GATE_TOLERANCE:.0f}% allowed\n")

## Run

Each stream downloads its two files, reads both parts, then deletes them. Both
files are saved after every stream, so a session that dies continues where it
stopped instead of starting the 16 GB again.

In [ ]:
def load(key):
    """Every row saved under `key`, or an empty list if no file exists."""
    if store.info(key) is None:
        return []
    text = store.download(key, CACHE).read_text(encoding="utf-8")
    return [json.loads(line) for line in text.splitlines() if line.strip()]


part1 = load(PART1_KEY)
part2 = load(PART2_KEY)
print(f"Resuming with {len(part1)} indicator readings and "
      f"{len(part2)} rescaling readings already saved.")

LOCAL1 = CACHE / "matched_statistics_partial.jsonl"
LOCAL2 = CACHE / "rescaling_matched_partial.jsonl"


def save():
    """Write both files and upload them. Called after every stream."""
    LOCAL1.write_text("".join(json.dumps(r) + "\n" for r in part1), encoding="utf-8")
    store.upload(LOCAL1, PART1_KEY, overwrite=True)
    LOCAL2.write_text("".join(json.dumps(r) + "\n" for r in part2), encoding="utf-8")
    store.upload(LOCAL2, PART2_KEY, overwrite=True)


# A stream counts as done only when it appears in BOTH files, so a session that
# stopped between the two parts redoes that stream rather than leaving half of
# it missing.
done = ({(r["order"], r["seed"]) for r in part1 if r["point"] == 10}
        & {(r["order"], r["seed"]) for r in part2})
if done:
    print("Streams already finished:", len(done), "of", len(READ_RUNS))

# The gate normally fires as soon as the fifth ordering A stream finishes. A
# session that resumes with ordering A already saved would skip those five
# streams and never reach it, so it is run here instead. That matters most
# exactly when it is easiest to forget: a resumed session is often a different
# Colab machine from the one that read ordering A.
gate_checked = False
if len({(r["order"], r["seed"]) for r in part1
        if r["order"] == "order_A" and r["point"] == 9}) == len(READ_SEEDS):
    check_gate(part1)
    gate_checked = True

began = time.perf_counter()
for index, (order, seed) in enumerate(READ_RUNS, start=1):
    if (order, seed) in done:
        continue
    # A restarted stream replaces its own rows instead of adding a second copy.
    part1 = [r for r in part1 if (r["order"], r["seed"]) != (order, seed)]
    part2 = [r for r in part2 if (r["order"], r["seed"]) != (order, seed)]

    stream_began = time.perf_counter()
    print(f"[{index}/{len(READ_RUNS)}] {order} seed {seed} ...", flush=True)
    paths = {}
    try:
        for point in READ_BOUNDARIES:
            if point == 0:
                continue
            key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
            paths[point] = store.download(key, CACHE)

        norms = {}
        for point in READ_BOUNDARIES:
            rows, norm = read_indicators(order, seed, point, paths.get(point))
            part1 += rows
            norms[point] = norm
        part2 += read_rescaling(order, seed, paths, norms)
    finally:
        for path in paths.values():
            Path(path).unlink(missing_ok=True)

    save()
    fixed = [r["effective_rank"] for r in part1
             if (r["order"], r["seed"]) == (order, seed) and r["point"] == 9
             and r["buffers"] == "start" and r["images"] == "in_stream"][0]
    matched = [r["effective_rank"] for r in part1
               if (r["order"], r["seed"]) == (order, seed) and r["point"] == 9
               and r["buffers"] == "own" and r["images"] == "in_stream"][0]
    print(f"    done in {(time.perf_counter() - stream_began) / 60:.1f} min   "
          f"boundary 9 effective rank: fixed {fixed:.3f}, matched {matched:.3f}   "
          f"weights shrank x{norms[10] / norms[9]:.3f} at the unlearn")

    # The gate fires once, as soon as all five ordering A streams are in hand.
    if not gate_checked:
        have_A = {(r["order"], r["seed"]) for r in part1
                  if r["order"] == "order_A" and r["point"] == 9}
        if len(have_A) == len(READ_SEEDS):
            check_gate(part1)
            gate_checked = True

print(f"\n{len(part1)} indicator readings and {len(part2)} rescaling readings "
      f"in {(time.perf_counter() - began) / 60:.1f} minutes.")

## Part 1, the table

The change over the stream is what the paper's correlation tables use. A
positive change on effective rank and srank, and a negative one on dormant
units, is what the plasticity story predicts.

In [ ]:
from statistics import mean, stdev

MEASURES = [("effective_rank", "Effective rank", "up"),
            ("srank", "srank", "up"),
            ("mean_share", "Mean row share", "."),
            ("dormant_0.01", "Dormant 0.01", "down"),
            ("dormant_0.025", "Dormant 0.025", "down"),
            ("dormant_0.1", "Dormant 0.1", "down")]


def pick(rows, **want):
    return [r for r in rows if all(r[k] == v for k, v in want.items())]


print("ALL FIFTEEN STREAMS, in-stream pictures, mean over streams (sd)")
print(f"{'Measure':<16} {'Route':<8} {'b0':>18} {'b9':>18} {'b10':>18}")
for field, label, _ in MEASURES:
    for mode, route in (("start", "fixed"), ("own", "matched")):
        line = f"{label:<16} {route:<8}"
        for point in (0, 9, 10):
            v = [r[field] for r in pick(part1, buffers=mode, point=point,
                                        images="in_stream")]
            line += f"   {mean(v):8.3f} ({stdev(v):6.3f})" if len(v) > 1 else "        ."
        print(line)

print()
print("DIRECTION OF TRAVEL over the nine groups, b0 to b9, per stream")
print(f"{'Measure':<16} {'Route':<8} {'predicted':>10} {'mean change':>13} "
      f"{'streams moving as predicted':>29}")
for field, label, want in MEASURES:
    if want == ".":
        continue
    for mode, route in (("start", "fixed"), ("own", "matched")):
        changes = []
        for order, seed in READ_RUNS:
            a = pick(part1, order=order, seed=seed, buffers=mode, point=0,
                     images="in_stream")
            b = pick(part1, order=order, seed=seed, buffers=mode, point=9,
                     images="in_stream")
            if a and b:
                changes.append(b[0][field] - a[0][field])
        if not changes:
            continue
        agree = sum((c > 0) == (want == "up") for c in changes)
        print(f"{label:<16} {route:<8} {want:>10} {mean(changes):+13.3f} "
              f"{agree:>20} of {len(changes)}")

## Part 2, the table

The question is how much of the rank change at the unlearn survives when the
weights are held at a constant size, and whether that changes once the
statistics are measured rather than assumed.

In [ ]:
print("THE RESCALING CONTROL, in-stream pictures, effective rank")
print("mean over the fifteen streams")
print(f"{'Route':<9} {'b9':>9} {'b10':>9} {'raw change':>12} "
      f"{'change at fixed size':>21} {'share from size':>17}")
for mode, route in (("start", "fixed"), ("own", "matched")):
    def arm(label):
        v = [r["effective_rank"] for r in pick(part2, buffers=mode, arm=label,
                                               images="in_stream")]
        return mean(v) if v else float("nan")

    b9, b10 = arm("b9"), arm("b10")
    held = arm("b10_at_b9_norm")
    raw = b10 - b9
    left = held - b9
    share = 1 - left / raw if raw else float("nan")
    print(f"{route:<9} {b9:9.3f} {b10:9.3f} {raw:+12.3f} {left:+21.3f} "
          f"{share:16.1%}")

print()
print("The weights at the unlearn, mean over streams:")
ratios = sorted({(r["order"], r["seed"]): r["norm_ratio"] for r in part2}.values())
print(f"  size after the unlearn, as a share of before: "
      f"{mean(ratios):.3f}, spread {min(ratios):.3f} to {max(ratios):.3f}")

## What this run decides

**Part 1.** If the matched route still shows the indicators moving the wrong
way across all fifteen streams, then sections 3.3 to 3.5 hold and the paper
can say so without the circularity it currently admits to. If the matched route
reverses them, those three sections are about the protocol and have to be
rewritten as such.

The direction table counts streams rather than averaging them, because fifteen
streams are really five starting points: the three orderings of one seed begin
from the same generator. A mean over fifteen correlated numbers overstates how
much evidence there is. Counting how many move as predicted does not.

**Part 2.** If the matched arms sit on top of each other while the fixed arms
separate, the rescaling result of notebook 46 was the fixed protocol seen from
another angle, and section 4.3 should say so. If the matched arms separate
too, the weight size is a real confound in its own right and the two thirds
figure stands.